# Notebook 09: Multi-Agent Systems — Supervisor, Swarm & Custom Architectures

## LangGraph 1.1.9

---

## Overview

This notebook covers **production multi-agent architectures** in LangGraph 1.1.9:

1. **Custom Coordinator** — Full-control patterns you build from scratch (extends Notebook 06)
2. **Supervisor Pattern** — Hierarchical systems via `langgraph-supervisor`
3. **Swarm Pattern** — Decentralized autonomous handoffs via `langgraph-swarm`
4. **Shared State & Communication** — How agents share information
5. **Error Propagation** — Building robust multi-agent pipelines
6. **Capstone Project** — A production-grade research system

---

## Prerequisites
- Completed Notebooks 01-08
- Familiarity with tool calling (Notebook 03), checkpointing (Notebook 04), HITL (Notebook 05), and parallel execution (Notebook 06)

## Estimated Time: 60-75 minutes
## Complexity Level: Expert (6/6)

---

## Installation

This notebook requires two optional LangGraph extension packages:

```bash
uv pip install langgraph-supervisor>=0.1.0 langgraph-swarm>=0.1.0
```

The core examples in Sections 4 and 6 run without these packages. Sections 5 and 8 require them.

## 1. Architecture Comparison

Before writing any code, choose the right pattern for your use case:

| Pattern | Routing Authority | Best For | Entry Point |
|---------|------------------|----------|-------------|
| **Custom Coordinator** | Central LLM — fully configurable | Domain-specific logic, tight state sharing | Build from scratch |
| **Supervisor** | Central LLM with structured handoffs | Enterprise-grade predictability, audit trails | `create_supervisor()` |
| **Swarm** | Each agent hands off autonomously | Speed + decentralized decision-making | `create_swarm()` |
| **Hierarchical** | Nested supervisors | Large orgs with departments / sub-teams | Compose supervisors |

---

### When to Use Each Pattern

**Use Supervisor when:**
- You need predictable, auditable routing
- Compliance or enterprise requirements mandate a clear decision trail
- You are building hierarchical org-chart-style workflows

**Use Swarm when:**
- Routing speed matters more than auditability
- Agents have enough context to self-route reliably
- You prefer decentralized, emergent decision-making

**Use Custom when:**
- You need non-standard routing logic (rules-based, ML-based, etc.)
- Agents share complex typed state beyond a message list
- The two packages above do not fit your schema

---

### Data Flow Diagrams

```
SUPERVISOR                       SWARM
─────────────────────────        ─────────────────────────
 User                             User
  │                                │
  ▼                                ▼
 Supervisor LLM ──► Agent A       Agent A ──► (transfer) ──► Agent B
  │           ◄── result            │                         │
  │                                 └──────────────────────── ▼
  └──────────► Agent B                                     Agent C
          ◄── result
  │
  ▼
 User response
```

## 2. Setup

In [1]:
import importlib.metadata
import os
from dotenv import load_dotenv

load_dotenv()

import langgraph
print(f"LangGraph: {importlib.metadata.version("langgraph")}")

try:
    import langgraph_supervisor
    print("langgraph-supervisor: installed")
    SUPERVISOR_AVAILABLE = True
except Exception:
    print("WARNING: langgraph-supervisor not installed. Run: uv pip install langgraph-supervisor>=0.1.0")
    SUPERVISOR_AVAILABLE = False

try:
    import langgraph_swarm
    print("langgraph-swarm: installed")
    SWARM_AVAILABLE = True
except Exception:
    print("WARNING: langgraph-swarm not installed. Run: uv pip install langgraph-swarm>=0.1.0")
    SWARM_AVAILABLE = False

if not os.getenv("NVIDIA_API_KEY"):
    raise ValueError("NVIDIA_API_KEY not set. Add it to your .env file.")

print("Setup complete")

LangGraph: 1.2.1
langgraph-supervisor: installed
langgraph-swarm: installed
Setup complete


In [2]:
import time

def _invoke_with_backoff(runnable, input_data, config=None, max_retries=5):
    """Invoke any LangChain runnable with exponential backoff on 429 rate-limit errors."""
    delay = 5
    for attempt in range(max_retries):
        try:
            if config:
                return runnable.invoke(input_data, config)
            return runnable.invoke(input_data)
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                print(f"⏳ Rate limited — retrying in {delay}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(delay)
                delay *= 2
            else:
                raise

print("✅ Backoff helper loaded")


✅ Backoff helper loaded


## 3. Custom Coordinator Pattern (No Extra Packages)

This extends the multi-agent pattern from Notebook 06 with a proper **coordinator node** that uses an LLM to decide routing. This is the foundation — understanding it makes Supervisor and Swarm easier to reason about.

### How It Works

```
START
  │
  ▼
coordinator (LLM decides next agent)
  ├──► researcher ──┐
  ├──► analyst   ──┤
  └──► writer    ──┘
                   │
                   ▼
                coordinator (review output, decide: done or route again)
                   │
                   ▼
                  END
```

Each node returns `Command(goto=...)` so the coordinator retains full control over the flow.

In [4]:
import operator
import json
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# ── State ────────────────────────────────────────────────────────────────────
class CoordinatorState(TypedDict):
    task: str                                         # Original user request
    messages: Annotated[list, operator.add]           # Full conversation log
    research_notes: Annotated[list[str], operator.add]  # Accumulated by researcher
    analysis_notes: Annotated[list[str], operator.add]  # Accumulated by analyst
    draft: str                                        # Written by writer (replaces)
    routing_history: Annotated[list[str], operator.add]  # Audit trail
    step_count: int

llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0)

# ── Coordinator node ─────────────────────────────────────────────────────────
ROUTING_SYSTEM = """
You are a coordinator managing a 3-person research team.
Given the current task and what has been done, decide the next step.

Available agents:
- researcher: gathers facts and information
- analyst: interprets findings and draws conclusions
- writer: produces the final written output
- FINISH: task is complete — return this when the writer has produced a draft

Rules:
1. Always research before analysing.
2. Always analyse before writing.
3. Return FINISH once the writer has produced output.
4. Never exceed 5 total steps.

Reply with ONLY one of: researcher | analyst | writer | FINISH
"""

def coordinator(state: CoordinatorState) -> Command:
    """LLM-driven coordinator — decides which agent to call next."""
    history_summary = ", ".join(state["routing_history"]) or "none yet"
    
    prompt = (
        f"Task: {state['task']}\n"
        f"Steps taken: {history_summary}\n"
        f"Research notes: {'yes' if state['research_notes'] else 'none'}\n"
        f"Analysis notes: {'yes' if state['analysis_notes'] else 'none'}\n"
        f"Draft produced: {'yes' if state['draft'] else 'no'}\n"
        f"Total steps so far: {state['step_count']}"
    )
    
    response = _invoke_with_backoff(llm, [
        SystemMessage(ROUTING_SYSTEM),
        HumanMessage(prompt),
    ])
    
    decision = response.content.strip().lower()
    print(f"[Coordinator] Step {state['step_count'] + 1} → {decision}")
    
    valid = {"researcher", "analyst", "writer", "finish"}
    if decision not in valid:
        decision = "finish"  # Safety fallback
    
    next_node = END if decision == "finish" else decision
    
    return Command(
        update={
            "routing_history": [decision],
            "step_count": state["step_count"] + 1,
        },
        goto=next_node,
    )

# ── Specialist nodes ──────────────────────────────────────────────────────────
def researcher(state: CoordinatorState) -> Command:
    """Gather facts — returns research notes and routes back to coordinator."""
    response = _invoke_with_backoff(llm, [
        SystemMessage("You are a researcher. Provide 3 concise bullet-point facts."),
        HumanMessage(f"Research: {state['task']}"),
    ])
    note = response.content
    print(f"[Researcher] Note: {note[:70]}...")
    return Command(
        update={"research_notes": [note]},
        goto="coordinator",
    )

def analyst(state: CoordinatorState) -> Command:
    """Analyse research notes — routes back to coordinator."""
    combined = "\n".join(state["research_notes"])
    response = _invoke_with_backoff(llm, [
        SystemMessage("You are an analyst. Provide 2 key insights from the research."),
        HumanMessage(f"Research:\n{combined}\n\nTask: {state['task']}"),
    ])
    insight = response.content
    print(f"[Analyst] Insight: {insight[:70]}...")
    return Command(
        update={"analysis_notes": [insight]},
        goto="coordinator",
    )

def writer(state: CoordinatorState) -> Command:
    """Produce a written summary — routes back to coordinator."""
    research = "\n".join(state["research_notes"])
    analysis = "\n".join(state["analysis_notes"])
    response = _invoke_with_backoff(llm, [
        SystemMessage("You are a technical writer. Write a clear 3-sentence executive summary."),
        HumanMessage(
            f"Task: {state['task']}\n\nResearch:\n{research}\n\nAnalysis:\n{analysis}"
        ),
    ])
    draft = response.content
    print(f"[Writer] Draft produced ({len(draft)} chars)")
    return Command(
        update={"draft": draft},
        goto="coordinator",
    )

# ── Build graph ────────────────────────────────────────────────────────────────
builder = StateGraph(CoordinatorState)
builder.add_node("coordinator", coordinator)
builder.add_node("researcher", researcher)
builder.add_node("analyst", analyst)
builder.add_node("writer", writer)

builder.add_edge(START, "coordinator")
# coordinator uses Command(goto=...) — no static edges needed from it
# researcher/analyst/writer also use Command — no static edges needed from them

custom_coordinator_app = builder.compile()
print("Custom coordinator graph compiled")

Custom coordinator graph compiled


In [5]:
# Run the custom coordinator
print("=" * 60)
print("Custom Coordinator — Multi-Agent Research")
print("=" * 60)

result = _invoke_with_backoff(custom_coordinator_app, {
    "task": "The impact of LangGraph on enterprise AI adoption in 2025",
    "messages": [],
    "research_notes": [],
    "analysis_notes": [],
    "draft": "",
    "routing_history": [],
    "step_count": 0,
})

print()
print("Routing path:", " -> ".join(result["routing_history"]))
print()
print("Final Draft:")
print("-" * 60)
print(result["draft"])

Custom Coordinator — Multi-Agent Research
[Coordinator] Step 1 → researcher
[Researcher] Note: Based on available data up to 2023, here are three concise bullet-poin...
[Coordinator] Step 2 → analyst
[Analyst] Insight: Based on the research, here are 2 key insights related to the potentia...
[Coordinator] Step 3 → writer
[Writer] Draft produced (714 chars)
[Coordinator] Step 4 → finish 

the task is complete, and the writer has produced a draft.

Routing path: researcher -> analyst -> writer -> finish

Final Draft:
------------------------------------------------------------
Here is a 3-sentence executive summary based on the research and analysis:

The introduction of LangGraph, a graph-based NLP framework, is poised to significantly enhance enterprise AI adoption in 2025 by improving the accuracy and efficiency of AI-powered language understanding. By facilitating the integration of diverse data sources and enabling the construction of comprehensive knowledge graphs, LangGraph is exp

## 4. Supervisor Pattern (`langgraph-supervisor`)

The `langgraph-supervisor` package provides `create_supervisor()`, which builds a standard LangGraph supervisor pattern without boilerplate.

### How the Supervisor Differs from a Custom Coordinator

| Aspect | Custom Coordinator | `create_supervisor()` |
|--------|-------------------|-----------------------|
| State schema | You define it | Uses `MessagesState` by default |
| Agent interface | Any callable | LangChain `Runnable` with `.name` |
| Routing | `Command(goto=...)` | Tool-call handoffs |
| Code required | ~60 lines | ~5 lines |
| Flexibility | Unlimited | Limited to package API |

### Supervisor Internals

Under the hood, `create_supervisor()` gives the supervisor LLM **handoff tools** — one per sub-agent. When the supervisor calls `transfer_to_researcher`, LangGraph routes execution to the researcher node. After the researcher finishes, control returns to the supervisor.

```
supervisor_llm
  │ calls tool: transfer_to_researcher
  ▼
researcher agent (runs fully)
  │ returns result to supervisor
  ▼
supervisor_llm
  │ calls tool: transfer_to_writer
  ▼
writer agent
  │
  ▼
supervisor_llm responds to user
```

In [8]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.tools import tool

# ── Simulated domain tools ────────────────────────────────────────────────────
@tool
def search_papers(query: str) -> str:
    """Search academic papers and return key findings."""
    return (
        f"Found 3 papers on '{query}':\n"
        "  1. 'Stateful LLM Workflows' (2025) — Key finding: graph-based state machines "
        "reduce hallucinations by 34%.\n"
        "  2. 'Agent Coordination at Scale' (2025) — Supervisor patterns outperform monolithic agents.\n"
        "  3. 'Enterprise AI Adoption' (2026) — 78% of Fortune 500 pilots use agentic frameworks."
    )

@tool
def analyze_trends(topic: str, data_points: str) -> str:
    """Analyze a set of data points and return trend insights."""
    return (
        f"Trend analysis for '{topic}' based on: {data_points[:80]}...\n"
        "  - Growth rate: 45% YoY\n"
        "  - Primary driver: cost reduction vs. human labor\n"
        "  - Risk: governance and compliance concerns remain top barrier"
    )

@tool
def write_executive_summary(title: str, findings: str) -> str:
    """Write a professional executive summary from research findings."""
    return (
        f"# {title}\n\n"
        f"**Executive Summary**\n\n"
        f"Based on current research, {findings[:120]}... "
        "The data indicates strong momentum with measurable ROI in early adopter organizations. "
        "Recommended action: begin a structured proof-of-concept within 90 days."
    )

print("Domain tools defined: search_papers, analyze_trends, write_executive_summary")

Domain tools defined: search_papers, analyze_trends, write_executive_summary


In [9]:
if not SUPERVISOR_AVAILABLE:
    print("SKIPPED — install langgraph-supervisor to run this section")
    print("  uv pip install langgraph-supervisor>=0.1.0")
else:
    from langgraph_supervisor import create_supervisor
    from langgraph.checkpoint.memory import MemorySaver

    llm_mini = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0)
    llm_smart = ChatNVIDIA(model="meta/llama-3.3-70b-instruct", temperature=0)

    # ── Sub-agents (each is a compiled graph with .name) ─────────────────────
    # create_supervisor expects agents that are compiled StateGraphs or
    # any LangChain Runnable — the simplest form is create_react_agent.
    from langgraph.prebuilt import create_react_agent

    research_agent = create_react_agent(
        llm_mini,
        tools=[search_papers],
        prompt=(
            "You are a research specialist. Use search_papers to find information. "
            "Always return structured findings with numbered points."
        ),
        name="researcher",
    )

    analysis_agent = create_react_agent(
        llm_mini,
        tools=[analyze_trends],
        prompt=(
            "You are a data analyst. Use analyze_trends to provide quantified insights. "
            "Focus on numbers, growth rates, and risk factors."
        ),
        name="analyst",
    )

    writing_agent = create_react_agent(
        llm_smart,
        tools=[write_executive_summary],
        prompt=(
            "You are a technical writer. Use write_executive_summary to produce "
            "polished, professional output suitable for C-suite executives."
        ),
        name="writer",
    )

    # ── Supervisor ────────────────────────────────────────────────────────────
    supervisor_app = create_supervisor(
        agents=[research_agent, analysis_agent, writing_agent],
        model=llm_smart,
        prompt=(
            "You are the research team supervisor. Coordinate three specialists:\n"
            "  - researcher: gathers academic papers and facts\n"
            "  - analyst: interprets data and identifies trends\n"
            "  - writer: produces executive-quality reports\n\n"
            "Always research first, then analyse, then write. "
            "Synthesize all results into a final response for the user."
        ),
    ).compile(checkpointer=MemorySaver())

    print("Supervisor compiled with 3 sub-agents")

    # ── Run ───────────────────────────────────────────────────────────────────
    config = {"configurable": {"thread_id": "supervisor-demo-001"}}
    print()
    print("=" * 60)
    print("Supervisor Pattern — Research Team")
    print("=" * 60)

    result = _invoke_with_backoff(supervisor_app, 
        {"messages": [{"role": "user", "content": (
            "Research the impact of LangGraph on enterprise AI adoption. "
            "Search for papers, analyse the trends, then write an executive summary."
        )}]},
        config,
    )

    print("\nFinal supervisor response:")
    print("-" * 60)
    print(result["messages"][-1].content)

/var/folders/zd/4xn4wx8n48q48lg906dk7k240000gn/T/ipykernel_3602/3895893300.py:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(
/var/folders/zd/4xn4wx8n48q48lg906dk7k240000gn/T/ipykernel_3602/3895893300.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  analysis_agent = create_react_agent(
/var/folders/zd/4xn4wx8n48q48lg906dk7k240000gn/T/ipykernel_3602/3895893300.py:36: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  writing_agent = create_react_agent(


Supervisor compiled with 3 sub-agents

Supervisor Pattern — Research Team

Final supervisor response:
------------------------------------------------------------
{"type": "function", "function": {"name": "transfer_to_researcher", "parameters": {}}}


## 5. Swarm Pattern (`langgraph-swarm`)

In a Swarm, there is **no central supervisor**. Each agent autonomously decides when to hand off to another agent using **transfer tools**. The conversation flows between agents based on their individual reasoning.

### Swarm vs. Supervisor — The Key Difference

```
SUPERVISOR (centralised):          SWARM (decentralised):
  User → Supervisor → Agent A        User → Agent A
          Supervisor → Agent B               Agent A: "I'll hand off to B"
          Supervisor → User          → Agent B
                                             Agent B: "Done — or transfer to C?"
                                     → User
```

### Handoff Mechanism

`langgraph-swarm` gives each agent a set of **transfer tools**. When an agent calls `transfer_to_technical_agent()`, the swarm routes execution to that agent. Under the hood this uses `Command(goto=..., update=...)` — the same primitive you used in Section 3.

```python
# What happens when an agent calls a transfer tool:
return Command(
    goto="technical_agent",
    update={"active_agent": "technical_agent"},
)
```

### Trade-offs

| | Supervisor | Swarm |
|---|---|---|
| Extra LLM call per routing | Yes — supervisor always decides | No — agent self-routes |
| Predictability | High | Lower — depends on agent reasoning |
| Audit trail | Clean — all routing via supervisor | Distributed — read `active_agent` history |
| Infinite loop risk | Low (supervisor can detect) | Higher — needs loop guards |

In [10]:
from langchain_core.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm_mini = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0)

# ── Domain tools ──────────────────────────────────────────────────────────────
@tool
def check_account_status(account_id: str) -> str:
    """Check account balance, status, and recent transactions."""
    return (
        f"Account {account_id}: Active | Balance: $1,234.56 | "
        "Last transaction: $45.00 on 2026-04-23 | No holds."
    )

@tool
def troubleshoot_api_error(error_code: str, service: str) -> str:
    """Diagnose and resolve API or technical errors."""
    return (
        f"Error {error_code} on {service}: Authentication token expired. "
        "Resolution: regenerate API keys in Settings > Security > API Keys. "
        "This resolves 92% of 401 errors."
    )

@tool
def process_refund_request(order_id: str, reason: str) -> str:
    """Process a billing refund or dispute."""
    return (
        f"Refund for order {order_id} (reason: {reason}): Approved for review. "
        "Estimated credit in 3-5 business days. Reference: REF-88321."
    )

print("Customer support tools defined")

Customer support tools defined


In [11]:
if not SWARM_AVAILABLE:
    print("SKIPPED — install langgraph-swarm to run this section")
    print("  uv pip install langgraph-swarm>=0.1.0")
else:
    from langgraph_swarm import create_swarm
    from langgraph.prebuilt import create_react_agent
    from langgraph.checkpoint.memory import MemorySaver

    # create_swarm adds transfer tools automatically based on the agent list.
    # Each agent gets: transfer_to_<other_agent_name> for every other agent.

    account_agent = create_react_agent(
        llm_mini,
        tools=[check_account_status],
        prompt=(
            "You are an account management specialist. Handle account inquiries. "
            "If the user has a technical/API issue, transfer to the technical agent. "
            "If the user has a billing dispute, transfer to the billing agent."
        ),
        name="account_agent",
    )

    technical_agent = create_react_agent(
        llm_mini,
        tools=[troubleshoot_api_error],
        prompt=(
            "You are a technical support engineer. Diagnose and resolve API and software errors. "
            "If the user has an account question, transfer to the account agent. "
            "If the user has a billing issue, transfer to the billing agent."
        ),
        name="technical_agent",
    )

    billing_agent = create_react_agent(
        llm_mini,
        tools=[process_refund_request],
        prompt=(
            "You are a billing specialist. Handle refunds, disputes, and payment issues. "
            "If the user has an account question, transfer to the account agent. "
            "If the user has a technical issue, transfer to the technical agent."
        ),
        name="billing_agent",
    )

    # create_swarm wires transfer tools automatically
    support_swarm = create_swarm(
        agents=[account_agent, technical_agent, billing_agent],
        default_active_agent="account_agent",  # First contact point
    ).compile(checkpointer=MemorySaver())

    print("Customer support swarm compiled")
    print("  - account_agent (default entry)")
    print("  - technical_agent")
    print("  - billing_agent")
    print("  Each agent can transfer to either of the other two autonomously.")

Customer support swarm compiled
  - account_agent (default entry)
  - technical_agent
  - billing_agent
  Each agent can transfer to either of the other two autonomously.


/var/folders/zd/4xn4wx8n48q48lg906dk7k240000gn/T/ipykernel_3602/254889128.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  account_agent = create_react_agent(
/var/folders/zd/4xn4wx8n48q48lg906dk7k240000gn/T/ipykernel_3602/254889128.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  technical_agent = create_react_agent(
/var/folders/zd/4xn4wx8n48q48lg906dk7k240000gn/T/ipykernel_3602/254889128.py:34: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  billing_agent = create_react_agent(


In [12]:
if not SWARM_AVAILABLE:
    print("SKIPPED — swarm not available")
else:
    # Test 1: Issue that starts with account agent and routes to technical
    config = {"configurable": {"thread_id": "swarm-demo-001"}}

    print("=" * 60)
    print("Swarm Test 1: Mixed issue (account + technical)")
    print("=" * 60)

    result = _invoke_with_backoff(support_swarm, 
        {"messages": [{"role": "user", "content": (
            "I cannot log in to my account — I keep getting a 401 API authentication error. "
            "My account ID is ACC-9981."
        )}]},
        config,
    )
    time.sleep(5)  # avoid 429 rate-limit

    print("\nFinal response:")
    print(result["messages"][-1].content)

    # Test 2: Pure billing issue — should stay with billing agent
    config2 = {"configurable": {"thread_id": "swarm-demo-002"}}

    print()
    print("=" * 60)
    print("Swarm Test 2: Billing dispute")
    print("=" * 60)

    result2 = _invoke_with_backoff(support_swarm, 
        {"messages": [{"role": "user", "content": (
            "I was charged twice for order ORD-5521 last week. "
            "I need a refund for the duplicate charge."
        )}]},
        config2,
    )

    print("\nFinal response:")
    print(result2["messages"][-1].content)

Swarm Test 1: Mixed issue (account + technical)

Final response:
Since the issue is a technical one, the user should be transferred to the technical agent.

Swarm Test 2: Billing dispute

Final response:
The billing agent will handle the dispute.


## 6. Inter-Agent Communication Patterns

Agents in a multi-agent system communicate in two fundamental ways:

### Method 1: Shared Typed State

All agents read from and write to a single `TypedDict` state. Use `Annotated[list, operator.add]` reducers so multiple agents can append without overwriting each other.

```python
class ProjectState(TypedDict):
    research_findings: Annotated[list[str], operator.add]  # agents append
    final_report: str                                       # last writer wins
```

**When to use:** Custom graphs with typed, domain-specific data (not just chat messages).

### Method 2: Message Passing

Agents communicate through the `messages` list using `AIMessage`, `HumanMessage`, and `ToolMessage`. This is what Supervisor and Swarm use internally.

```python
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # all agents append here
```

**When to use:** Conversational agents, LangChain tool-calling agents, Supervisor/Swarm patterns.

---

The cell below demonstrates **shared typed state** — each agent appends its results to a shared list, and the next agent reads from it.

In [13]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage, SystemMessage

# ── Shared typed state with reducers ──────────────────────────────────────────
class SharedProjectState(TypedDict):
    project_brief: str
    research_findings: Annotated[list[str], operator.add]  # Multiple agents append
    analysis_results: Annotated[list[str], operator.add]
    draft_sections: Annotated[list[str], operator.add]
    final_report: str
    status: str

llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0)

# ── Agent nodes — each reads prior state and appends its output ───────────────
def research_agent(state: SharedProjectState) -> dict:
    """Research agent — appends to shared research_findings."""
    prompt = f"Provide 2 concise facts about: {state['project_brief']}"
    response = _invoke_with_backoff(llm, [
        SystemMessage("You are a researcher. Be factual and concise."),
        HumanMessage(prompt),
    ])
    finding = response.content
    print(f"Research agent: {finding[:70]}...")
    return {"research_findings": [finding]}

def analysis_agent(state: SharedProjectState) -> dict:
    """Analyst — reads research_findings, appends analysis_results."""
    findings_text = "\n".join(state["research_findings"])
    prompt = f"Identify 2 key insights from:\n{findings_text}"
    response = _invoke_with_backoff(llm, [
        SystemMessage("You are an analyst. Be precise."),
        HumanMessage(prompt),
    ])
    analysis = response.content
    print(f"Analysis agent: {analysis[:70]}...")
    return {"analysis_results": [analysis]}

def writing_agent(state: SharedProjectState) -> dict:
    """Writer — reads all prior work, produces final_report."""
    context = (
        f"Project: {state['project_brief']}\n"
        f"Research: {'; '.join(state['research_findings'])}\n"
        f"Analysis: {'; '.join(state['analysis_results'])}"
    )
    prompt = f"Write a 3-sentence executive summary from:\n{context}"
    response = _invoke_with_backoff(llm, [
        SystemMessage("You are a technical writer."),
        HumanMessage(prompt),
    ])
    report = f"# Project Report: {state['project_brief']}\n\n{response.content}"
    print(f"Writing agent: report generated ({len(report)} chars)")
    return {"final_report": report, "status": "complete"}

# ── Sequential pipeline with shared state ─────────────────────────────────────
builder = StateGraph(SharedProjectState)
builder.add_node("researcher", research_agent)
builder.add_node("analyst", analysis_agent)
builder.add_node("writer", writing_agent)

builder.add_edge(START, "researcher")
builder.add_edge("researcher", "analyst")
builder.add_edge("analyst", "writer")
builder.add_edge("writer", END)

pipeline = builder.compile()

# ── Run ───────────────────────────────────────────────────────────────────────
print("=" * 60)
print("Shared State Pipeline")
print("=" * 60)

result = pipeline.invoke({
    "project_brief": "LangGraph in enterprise AI development — 2026 outlook",
    "research_findings": [],
    "analysis_results": [],
    "draft_sections": [],
    "final_report": "",
    "status": "in_progress",
})

print()
print("=" * 60)
print(result["final_report"])
print(f"\nStatus: {result['status']}")
print(f"Research entries: {len(result['research_findings'])}")
print(f"Analysis entries: {len(result['analysis_results'])}")

Shared State Pipeline
Research agent: Based on available data up to 2023, here are two concise facts about L...
Analysis agent: Based on the provided information, here are 2 key insights:

1. **Lang...
Writing agent: report generated (680 chars)

# Project Report: LangGraph in enterprise AI development — 2026 outlook

Here is a 3-sentence executive summary:

LangGraph, a graph-based language model, is poised to revolutionize enterprise AI development by enabling more accurate and efficient AI applications, particularly in natural language processing (NLP) tasks. By 2026, LangGraph is expected to be integrated with edge AI technologies, allowing for real-time processing of AI workloads at the edge of the network and expanding its applications to resource-constrained environments. This integration will unlock new possibilities for AI deployment in IoT devices, autonomous vehicles, and other edge computing use cases.

Status: complete
Research entries: 1
Analysis entries: 1


## 7. Error Propagation in Multi-Agent Systems

When one agent in a multi-agent system fails, different architectures handle errors differently:

| Architecture | Error Behavior | Recovery Option |
|---|---|---|
| Supervisor | Supervisor sees the error via ToolMessage and can re-route | Easy — supervisor re-calls a different agent |
| Swarm | Agent exception may leave state inconsistent | Hard — requires per-agent try/except |
| Custom | You control fully | Use `Command(goto='retry')` or `Command(goto=END)` |

### Best Practices

1. **Always catch exceptions** in agent nodes — never let an unhandled exception kill the graph
2. **Use Command to route to a retry node** — combine with `step_count` guards to avoid infinite loops
3. **Store errors in state** — makes debugging and auditing easier
4. **Degrade gracefully** — if retries are exhausted, route to END with a user-friendly error message

The pattern below demonstrates retry-with-fallback using `Command`.

In [14]:
import random
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage

class RobustState(TypedDict):
    task: str
    result: str
    error_log: list[str]
    attempts: int
    max_attempts: int

llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0)

MAX_RETRIES = 3

def unreliable_agent(state: RobustState) -> Command:
    """
    Agent that simulates intermittent failures.
    On the first two attempts it raises; on the third it succeeds.
    In production, failures come from network errors, rate limits, etc.
    """
    attempt = state["attempts"]
    print(f"[Agent] Attempt {attempt + 1}/{state['max_attempts']}")

    try:
        # Simulate failure on first 2 attempts
        if attempt < 2:
            raise RuntimeError(f"Simulated transient error on attempt {attempt + 1}")

        response = _invoke_with_backoff(llm, [HumanMessage(f"Answer briefly: {state['task']}")])
        print(f"[Agent] Success on attempt {attempt + 1}")
        return Command(
            update={"result": response.content, "attempts": attempt + 1},
            goto=END,
        )

    except Exception as exc:
        error_msg = f"Attempt {attempt + 1}: {exc}"
        print(f"[Agent] FAILED — {error_msg}")

        new_attempts = attempt + 1
        if new_attempts >= state["max_attempts"]:
            # Exhausted retries — degrade gracefully
            print("[Agent] Max retries reached — returning fallback response")
            return Command(
                update={
                    "result": "Service temporarily unavailable. Please try again later.",
                    "error_log": [error_msg],
                    "attempts": new_attempts,
                },
                goto=END,
            )

        # Retry available — route to wait node
        return Command(
            update={"error_log": [error_msg], "attempts": new_attempts},
            goto="retry_handler",
        )

def retry_handler(state: RobustState) -> Command:
    """
    Retry handler — in production you would add exponential backoff here
    (time.sleep(2 ** state['attempts'])) or switch to an alternative service.
    """
    print(f"[Retry] Preparing attempt {state['attempts'] + 1}...")
    # In production: time.sleep(2 ** state['attempts'])
    return Command(goto="agent")

# ── Build graph ────────────────────────────────────────────────────────────────
builder = StateGraph(RobustState)
builder.add_node("agent", unreliable_agent)
builder.add_node("retry_handler", retry_handler)

builder.add_edge(START, "agent")
# agent and retry_handler use Command — no static edges needed from them

robust_app = builder.compile()

print("=" * 60)
print("Error Propagation & Retry Demonstration")
print("=" * 60)
print()

result = _invoke_with_backoff(robust_app, {
    "task": "What is the capital of France?",
    "result": "",
    "error_log": [],
    "attempts": 0,
    "max_attempts": MAX_RETRIES,
})

print()
print("Final result:", result["result"])
print("Total attempts:", result["attempts"])
if result["error_log"]:
    print("Error log:")
    for entry in result["error_log"]:
        print(f"  - {entry}")

Error Propagation & Retry Demonstration

[Agent] Attempt 1/3
[Agent] FAILED — Attempt 1: Simulated transient error on attempt 1
[Retry] Preparing attempt 2...
[Agent] Attempt 2/3
[Agent] FAILED — Attempt 2: Simulated transient error on attempt 2
[Retry] Preparing attempt 3...
[Agent] Attempt 3/3
[Agent] Success on attempt 3

Final result: The capital of France is Paris.
Total attempts: 3
Error log:
  - Attempt 2: Simulated transient error on attempt 2


## 8. Capstone Project: Production Multi-Agent Research System

Combine everything from this notebook into a production-grade system:

- **Supervisor pattern** coordinates three domain-expert agents
- **SqliteSaver** provides persistent memory across sessions
- **Error handling** in each agent node
- **Human-in-the-loop gate** before publishing the final report (bonus)

### System Architecture

```
User Input
    │
    ▼
Research Supervisor (meta/llama-3.3-70b-instruct)
    │
    ├── transfer_to_technical_researcher ──► Technical Researcher
    │                                            │ (search_technical_papers)
    │                                            │
    ├── transfer_to_industry_analyst ────────────► Industry Analyst
    │                                            │ (search_industry_reports)
    │                                            │
    └── transfer_to_synthesis_agent ─────────────► Synthesis Agent
                                                 │ (synthesize_research)
                                                 │
                                          Supervisor synthesises
                                                 │
                                              User
```

### Why This Matters in Production

- Each specialist runs a smaller, cheaper model (`meta/llama-3.1-8b-instruct`)
- The supervisor uses a smarter model (`meta/llama-3.3-70b-instruct`) only for coordination
- SqliteSaver enables resumable sessions — if a task fails mid-way, re-invoke with the same `thread_id`
- The modular structure makes it easy to swap or add specialists without changing routing logic

In [15]:
from langchain_core.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# ── Specialist tools ──────────────────────────────────────────────────────────
@tool
def search_technical_papers(query: str) -> str:
    """Search academic and technical papers on a given topic."""
    return (
        f"Technical search for '{query}':\n"
        "  Paper 1: 'Multi-Agent LLM Architectures' (2026) — "
        "Supervisor patterns reduce coordination latency by 28%.\n"
        "  Paper 2: 'LangGraph at Scale' (2025) — "
        "Stateful agents outperform stateless chains for complex tasks.\n"
        "  Paper 3: 'Enterprise AI Benchmarks' (2026) — "
        "Agentic systems show 3.2x better task completion rates."
    )

@tool
def search_industry_reports(query: str) -> str:
    """Search market research and industry analyst reports."""
    return (
        f"Industry reports for '{query}':\n"
        "  Gartner (2026): Agentic AI market $12B, 67% YoY growth.\n"
        "  Forrester (2026): 54% of enterprises running pilot agentic workflows.\n"
        "  McKinsey (2026): ROI of $2.40 per $1 invested in multi-agent systems."
    )

@tool
def synthesize_research(technical_findings: str, industry_findings: str, topic: str) -> str:
    """Synthesize technical and industry research into a coherent analysis."""
    return (
        f"Synthesis for '{topic}':\n\n"
        f"Technical perspective: {technical_findings[:120]}...\n\n"
        f"Market perspective: {industry_findings[:120]}...\n\n"
        "Conclusion: Both academic evidence and market data confirm strong adoption momentum. "
        "The technology is mature enough for production deployment with appropriate governance."
    )

print("Specialist tools defined")

Specialist tools defined


In [18]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

if not SUPERVISOR_AVAILABLE:
    print("Capstone demo requires langgraph-supervisor")
    print("  uv pip install langgraph-supervisor>=0.1.0")
    print()
    print("Showing custom-coordinator fallback instead...")

    # ── Fallback: custom coordinator using Section 3 pattern ──────────────────
    from langgraph.graph import StateGraph, START, END
    from langgraph.types import Command
    from langchain_core.messages import SystemMessage, HumanMessage
    from langchain_nvidia_ai_endpoints import ChatNVIDIA
    import operator
    from typing import Annotated
    from typing_extensions import TypedDict

    class ProductionState(TypedDict):
        topic: str
        technical: Annotated[list[str], operator.add]
        industry: Annotated[list[str], operator.add]
        synthesis: str
        phase: str

    llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0)

    def tech_researcher(state: ProductionState) -> dict:
        result = search_technical_papers.invoke({"query": state["topic"]})
        time.sleep(5)  # avoid 429 rate-limit
        print(f"[Tech Researcher] {result[:80]}...")
        return {"technical": [result], "phase": "industry_research"}

    def industry_analyst_node(state: ProductionState) -> dict:
        result = search_industry_reports.invoke({"query": state["topic"]})
        time.sleep(5)  # avoid 429 rate-limit
        print(f"[Industry Analyst] {result[:80]}...")
        return {"industry": [result], "phase": "synthesis"}

    def synthesizer_node(state: ProductionState) -> dict:
        result = synthesize_research.invoke({
            "technical_findings": "\n".join(state["technical"]),
            "industry_findings": "\n".join(state["industry"]),
            "topic": state["topic"],
        })
        print(f"[Synthesizer] Report generated")
        return {"synthesis": result, "phase": "complete"}

    def router(state: ProductionState) -> str:
        return state["phase"]

    gb = StateGraph(ProductionState)
    gb.add_node("tech_research", tech_researcher)
    gb.add_node("industry_research", industry_analyst_node)
    gb.add_node("synthesis", synthesizer_node)
    gb.add_edge(START, "tech_research")
    gb.add_edge("tech_research", "industry_research")
    gb.add_edge("industry_research", "synthesis")
    gb.add_edge("synthesis", END)

    db = sqlite3.connect(":memory:", check_same_thread=False)
    memory = SqliteSaver(db)
    production_app = gb.compile(checkpointer=memory)

    config = {"configurable": {"thread_id": "production-fallback-001"}}
    result = _invoke_with_backoff(production_app, {
        "topic": "LangGraph adoption in enterprise AI systems",
        "technical": [],
        "industry": [],
        "synthesis": "",
        "phase": "tech_research",
    }, config)

    print()
    print("=" * 60)
    print(result["synthesis"])

else:
    from langgraph_supervisor import create_supervisor
    from langgraph.prebuilt import create_react_agent

    llm_fast = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0).with_retry(
        stop_after_attempt=5,
        wait_exponential_jitter=True
    )
    llm_smart = ChatNVIDIA(model="meta/llama-3.3-70b-instruct", temperature=0).with_retry(
        stop_after_attempt=5,
        wait_exponential_jitter=True
    )

    # ── Specialist agents ─────────────────────────────────────────────────────
    tech_researcher = create_react_agent(
        llm_fast,
        tools=[search_technical_papers],
        prompt="You are a technical research specialist. Search for academic papers and return structured findings.",
        name="technical_researcher",
    )

    industry_analyst = create_react_agent(
        llm_fast,
        tools=[search_industry_reports],
        prompt="You are an industry analyst. Search for market data and business insights.",
        name="industry_analyst",
    )

    synthesis_expert = create_react_agent(
        llm_smart,
        tools=[synthesize_research],
        prompt=(
            "You are a research synthesizer. Combine technical and industry findings. "
            "Always call synthesize_research with both technical_findings and industry_findings."
        ),
        name="synthesis_agent",
    )

    # ── Production supervisor with SQLite persistence ─────────────────────────
    db = sqlite3.connect("enterprise_research.db", check_same_thread=False)
    memory = SqliteSaver(db)

    research_system = create_supervisor(
        agents=[tech_researcher, industry_analyst, synthesis_expert],
        model=llm_smart,
        prompt=(
            "You coordinate a three-person research team:\n"
            "  - technical_researcher: for academic papers and technical facts\n"
            "  - industry_analyst: for market data and business insights\n"
            "  - synthesis_agent: for combining findings into a final analysis\n\n"
            "Workflow: always run technical_researcher first, then industry_analyst, "
            "then synthesis_agent. After synthesis, provide a final executive response "
            "to the user."
        ),
    ).compile(checkpointer=memory)

    config = {"configurable": {"thread_id": "production-research-001"}}

    print("=" * 60)
    print("Capstone: Production Multi-Agent Research System")
    print("=" * 60)

    result = _invoke_with_backoff(
        research_system,
        {"messages": [{"role": "user", "content": (
            "Research the current state of LangGraph adoption in enterprise AI. "
            "Gather both technical and industry perspectives, then synthesize the findings."
        )}]},
        config,
    )

    print("\nFinal synthesis:")
    print("-" * 60)
    print(result["messages"][-1].content)

    # ── Resume same session (demonstrates persistence) ────────────────────────
    print()
    print("=" * 60)
    print("Follow-up question (same thread — state persists)")
    print("=" * 60)

    result2 = _invoke_with_backoff(
        research_system,
        {"messages": [{"role": "user", "content": 
            "Based on your research, what are the top 2 risks for enterprise adoption?"}]},
        config,  # Same thread_id — supervisor remembers previous messages
    )

    print("\nFollow-up response:")
    print(result2["messages"][-1].content)


/var/folders/zd/4xn4wx8n48q48lg906dk7k240000gn/T/ipykernel_3602/620473469.py:86: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  tech_researcher = create_react_agent(
/var/folders/zd/4xn4wx8n48q48lg906dk7k240000gn/T/ipykernel_3602/620473469.py:93: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  industry_analyst = create_react_agent(
/var/folders/zd/4xn4wx8n48q48lg906dk7k240000gn/T/ipykernel_3602/620473469.py:100: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  synthesis_expert = create_react_agent

Capstone: Production Multi-Agent Research System


Exception: [429] Too Many Requests
{'status': 429, 'title': 'Too Many Requests'}

## 9. Exercises

---

### Exercise 1: Convert Custom Coordinator to Supervisor

Take the custom coordinator from **Section 3** and recreate it using `create_supervisor()`. Things to observe:

- How much code is eliminated?
- Does routing behaviour change? (Test with the same prompt)
- What do you lose in terms of control over the `CoordinatorState` fields?

```python
# Starter — fill in the blanks
from langgraph_supervisor import create_supervisor
from langgraph.prebuilt import create_react_agent

research_agent = create_react_agent(llm, tools=[...], name="researcher")
analysis_agent = create_react_agent(llm, tools=[...], name="analyst")
writing_agent  = create_react_agent(llm, tools=[...], name="writer")

supervisor = create_supervisor(
    agents=[research_agent, analysis_agent, writing_agent],
    model=...,
    prompt=...,
).compile()
```

---

### Exercise 2: Build a 4-Agent Customer Support Swarm

Create a swarm with:
- `intake_agent`: First contact — greets the user and identifies issue type, then routes
- `billing_agent`: Handles payment issues and refunds
- `technical_agent`: Handles software and API problems
- `escalation_agent`: Handles VIP customers and unresolved issues

Each agent should transfer to any other when appropriate. Test with at least 3 different user messages that exercise different routing paths.

---

### Exercise 3: Add Human-in-the-Loop to the Supervisor

Modify the capstone research system to pause before the synthesis step and ask a human: "Research complete. Proceed with synthesis? (yes/no)"

If the human replies "yes", run synthesis. If "no", end with just the raw research.

<details>
<summary>Exercise 3 Hint</summary>

Add a review gate node between your research agents and the synthesizer. Use `interrupt()` to pause execution and `Command(resume=...)` to continue:

```python
from langgraph.types import interrupt, Command

def human_review_gate(state) -> Command:
    """Pause and ask for human approval before synthesis."""
    # interrupt() saves state to the checkpointer and pauses the graph.
    # The value you pass is returned to the caller of .invoke() / .stream().
    approval = interrupt({
        "question": "Research gathered. Proceed with synthesis?",
        "research_summary": state["messages"][-1].content,
    })
    
    if approval.lower() == "yes":
        return Command(goto="synthesis_agent")
    
    return Command(
        goto=END,
        update={"messages": [{"role": "assistant", "content": "Synthesis cancelled by reviewer."}]},
    )

# To resume after interrupt:
# result = app.invoke(Command(resume="yes"), config)
```

Remember: the checkpointer **must** be configured (`SqliteSaver` or `MemorySaver`) for `interrupt()` to work.
</details>

---

### Exercise 4: Parallel Agent Execution with Supervisor

The default supervisor calls agents **sequentially**. Modify the capstone so the `technical_researcher` and `industry_analyst` run **in parallel** (using `Send`), and only the `synthesis_agent` runs after both complete.

Hint: You will need to move away from `create_supervisor()` and back to a custom graph — this is a deliberate trade-off between convenience and control.

## 10. Summary

---

### Patterns Covered

**Custom Coordinator** (Section 3):
- Build with `StateGraph` + `Command(goto=...)` nodes
- Full control over routing logic, state shape, and error handling
- Best when you need domain-specific routing or typed state beyond messages

**Supervisor Pattern** (`langgraph-supervisor`, Section 4):
- Central LLM decides which sub-agent to call via tool-call handoffs
- Highly predictable and auditable — every routing decision is logged
- Slightly slower: one extra LLM call per routing step
- `create_supervisor(agents, model, prompt)` — minimal boilerplate

**Swarm Pattern** (`langgraph-swarm`, Section 5):
- Each agent decides autonomously when to transfer to another agent
- Faster: no separate routing LLM call
- Less predictable: requires careful system-prompt engineering per agent
- `create_swarm(agents, default_active_agent)` — wires transfer tools automatically

**Shared State Communication** (Section 6):
- `Annotated[list, operator.add]` reducers let multiple agents append without overwriting
- Best for typed, structured data shared across a pipeline

**Error Propagation** (Section 7):
- Use `Command(goto='retry_handler')` for retryable errors
- Use `Command(goto=END, update={...})` for graceful degradation
- Always bound retry loops with a step counter

---

### Production Checklist for Multi-Agent Systems

- Use `SqliteSaver` or `PostgresSaver` (not `MemorySaver`) for any system that needs to survive restarts
- Enable LangSmith tracing — multi-agent routing decisions are hard to debug without it
- Add `interrupt()` gates before high-stakes, irreversible actions (sending emails, publishing, charging cards)
- Guard every retry loop with a `max_attempts` counter in state
- Use smaller/cheaper models for specialist agents and smarter models only for supervisors
- Test each agent in isolation before wiring them together

---

### Decision Tree

```
Do you need custom typed state (beyond messages)?
  YES → Custom Coordinator (Section 3)
  NO  →
      Is predictable, auditable routing important?
        YES → Supervisor (langgraph-supervisor)
        NO  →
            Do agents have enough context to self-route reliably?
              YES → Swarm (langgraph-swarm)
              NO  → Supervisor (safer default)
```

---

## Next Steps

**Notebook 10: Testing, Observability & LangGraph Platform**

- Unit-testing graph nodes and edges
- Integration testing multi-agent flows with `MemorySaver`
- LangSmith tracing and evaluation dashboards
- Deploying to LangGraph Platform (Cloud or self-hosted)
- LangGraph Studio v2 — visual debugging for multi-agent graphs